# Agent Network: Typed Peer Handoffs Without Ping-Pong

| Field | Value |
|---|---|
| Stage | Multi-agent RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A network needs typed handoffs, an owner, and a hop budget; peer freedom is not permission to loop.

## 30-Second Summary

A policy specialist retrieves a retention fact, hands a typed calculation request to a math specialist, and receives a derived weekly value with provenance.

## Why This Matters

Peer-to-peer coordination fits fluid expertise, but unrestricted handoffs create loops, duplicated work, and ambiguous answer ownership.

## Scope

| Covers | Does not cover |
|---|---|
| Typed envelope, peer handoff, hop budget, provenance, terminal owner | Dynamic LLM routing, concurrent messaging, distributed transport |


## Mental Model

```text
policy peer --evidence+request--> math peer --derived result--> policy owner -> answer
```


In [1]:
POLICY = {"source": "retention-policy", "days": 28}
request = {"question": "How many weeks are logs retained?", "owner": "policy", "hops": 0, "max_hops": 2}


## How It Works

The policy peer owns the user answer. It emits a handoff with a schema, evidence citation, and remaining budget. The math peer returns only the requested derived value.


## Baseline

An untyped message loses the source and gives the receiving peer no stopping contract.


In [2]:
baseline_message = "Retention is 28 days. Convert it."
baseline_has_provenance = "source" in baseline_message.lower()
baseline_has_provenance


False

## Technique Implementation

The typed envelope retains provenance and increments the hop count exactly once per boundary crossing.


In [3]:
def policy_peer(state: dict) -> dict:
    assert state["hops"] < state["max_hops"]
    return {**state, "hops": state["hops"] + 1, "to": "math", "operation": "days_to_weeks", "evidence": POLICY}

def math_peer(message: dict) -> dict:
    assert message["operation"] == "days_to_weeks"
    return {"to": message["owner"], "weeks": message["evidence"]["days"] / 7, "source": message["evidence"]["source"], "hops": message["hops"] + 1}

handoff = policy_peer(request)
result = math_peer(handoff)
result


{'to': 'policy', 'weeks': 4.0, 'source': 'retention-policy', 'hops': 2}

## Controlled Experiment

We require a correct derived value, preserved source, return to the original owner, and no hop-budget overflow.


In [4]:
answer = f"Logs are retained for {result['weeks']:.0f} weeks [{result['source']}]."
experiment = {"answer": answer, "owner_restored": result["to"] == request["owner"], "within_budget": result["hops"] <= request["max_hops"]}
experiment


{'answer': 'Logs are retained for 4 weeks [retention-policy].',
 'owner_restored': True,
 'within_budget': True}

## Evaluation

The network completes in **2 hops**, returns to the policy owner, and preserves the source. The baseline exposes neither provenance nor a stopping rule.


In [5]:
assert not baseline_has_provenance
assert result == {"to": "policy", "weeks": 4.0, "source": "retention-policy", "hops": 2}
assert experiment["owner_restored"] and experiment["within_budget"]
print("Agent-network checks passed.")


Agent-network checks passed.


## Decision Guide

| Need | Choice |
|---|---|
| Fluid peer expertise | Network |
| Stable central routing | Supervisor |
| Organizational subteams | Hierarchy |
| No cross-skill dependency | Single agent |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Ping-pong | Peers can re-delegate forever | Hop budget and visited roles |
| Lost citation | String-only handoff | Typed evidence envelope |
| No final answer | No terminal owner | Preserve owner field |
| Duplicate work | Roles overlap | Capability contracts |


## Production Notes

### Observability
Record handoff ID, roles, operation, evidence IDs, hop count, and terminal owner.

### Safety and Guardrails
Each peer receives least-privilege tools and data.

### Latency and Cost
Use direct calls for known dependencies; avoid a network for one skill.


## Practice

Add a currency-conversion peer and reject a third hop with a deterministic terminal reason.

## Recall

Toggle - Recall: What prevents ping-pong?
A hop budget, visited-role guard, and terminal owner.

Toggle - Recall: Why type handoffs?
To preserve operation, evidence, provenance, and accountability.

## Sources

- [LangChain handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the bounded handoff contract | Add peer failure and timeout handling |
